# Mini-GPT Walkthrough

## Goal / Mục tiêu
Quan sát tokenizer, next-token batch và một training step nhỏ, chạy được trên CPU.

## Setup / Chuẩn bị
Notebook chỉ cần PyTorch. Seed cố định giúp kết quả tái lập gần đúng.

In [ ]:
import random
import torch

SEED = 3407
random.seed(SEED)
torch.manual_seed(SEED)
print({'torch': torch.__version__, 'device': 'cpu'})


## Steps / Các bước
### 1. Tokenize và tạo next-token pairs

In [ ]:
text = 'học máy từ dữ liệu. ' * 8
vocab = sorted(set(text))
stoi = {char: index for index, char in enumerate(vocab)}
itos = {index: char for char, index in stoi.items()}
tokens = torch.tensor([stoi[char] for char in text])
block_size = 8
x = tokens[:block_size][None, :]
y = tokens[1:block_size + 1][None, :]
print({'vocab': len(vocab), 'x_shape': tuple(x.shape), 'x': x.tolist(), 'y': y.tolist()})


### 2. Train một bigram neural baseline

In [ ]:
model = torch.nn.Embedding(len(vocab), len(vocab))
optimizer = torch.optim.AdamW(model.parameters(), lr=0.05)
losses = []
for _ in range(40):
    logits = model(x)
    loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), y.flatten())
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
print({'initial_loss': round(losses[0], 4), 'final_loss': round(losses[-1], 4)})


## Checks / Kiểm tra

In [ ]:
assert x.shape == y.shape == (1, block_size)
assert losses[-1] < losses[0]
assert ''.join(itos[i] for i in tokens[:5].tolist()) == text[:5]
print('PASS')


## Next Steps / Bước tiếp
Thay bigram Embedding bằng `MiniGPT` trong `../python/mini_gpt_lab.py`; so sánh parameter count và validation loss.